# 二重三相IPMSMのトルク測定値予測

## データ
二重三相永久磁石同期モータ（DTP-IPMSM）の実験データを使用する。
本データの対象試験機であるDTP-IPMSMの各巻線系統ごとに電源，逆流防止ダイオードおよび三相インバータを構成しており，
第2巻線にスイッチを付加し，この開閉によりDC電源を即断して故障を模擬したものである。
データの実験条件は Vdc1=140V（健全駆動），Vdc2=0V（故障駆動） とし，第2巻線のDC電源故障時に提案制御法を適用している。

入力変数
- τ1*（第1系統トルク指令値）
- ^τ1（第1系統トルク推定値）
- Iq2（第2系統トルクに寄与する電流）
- ω_m（機械角速度）
- Id1（第1系統磁束に寄与する電流）
- Iq1（第1系統トルクに寄与する電流）
- τ2*（第2系統トルク指令値）
- ^τ2（第2系統トルク推定値）
- Vdc1（DC電源1）
- Vdc2（DC電源2）
- Id2（第2系統磁束に寄与する電流）

目的変数
- トルク測定値
AutoMLによりトルク測定に有効性があるかを検証する。

## 使用手法
PyCaretによるAutoML（Regression）

## 結果
PyCaretによるAutoMLを用いて複数の回帰モデルを比較した結果，
Orthogonal Matching Pursuitが最も良い性能を示した。

- RMSE：0.2201
- MAE：0.1179
- R²：0.0732

本結果より，本実験データからトルク測定値を予測する回帰モデルは構築できたものの，
決定係数R²が0.0732と低く，今回使用した説明変数のみではトルク測定値を十分に説明できないことがわかった。

In [1]:
import pandas as pd
data = pd.read_excel(r"C:\Users\ogara\OneDrive\Desktop\負荷75重みVdc比.xlsx")

In [2]:
data = pd.read_excel(
    r"C:\Users\ogara\OneDrive\Desktop\負荷75重みVdc比.xlsx",
    header=18,
    skiprows=[19]
)

data.columns = [
    "Time",
    "TorqueCmd1",
    "TorqueEst1",
    "Iq2",
    "Speed",
    "Id1",
    "Iq1",
    "TorqueMeasured",
    "TorqueCmd2",
    "TorqueEst2",
    "Vdc1",
    "Vdc2",
    "Id2"
]

In [3]:
data.head()

,Time,TorqueCmd1,TorqueEst1,Iq2,Speed,Id1,Iq1,TorqueMeasured,TorqueCmd2,TorqueEst2,Vdc1,Vdc2,Id2
0,-0.249,1.263750,1.216875,0.736875,-0.023125,0.367500,1.010625,1.718750,1.250625,0.895000,2.035000,2.014375,-0.491875
1,-0.248,1.253750,0.875000,0.985625,0.208750,-0.636250,0.716875,1.747500,1.240625,1.190625,2.034375,2.013125,0.390000
2,-0.247,1.260000,1.109375,0.816875,0.073125,0.268750,0.915625,1.777500,1.246875,0.995625,2.033125,2.013750,-0.480625
3,-0.246,1.254375,1.128750,0.825625,0.168750,-0.219375,0.930625,1.791875,1.241875,0.999375,2.036875,2.016875,0.069375
4,-0.245,1.260000,1.198125,0.888750,0.021250,-0.363125,0.986250,1.793125,1.246875,1.073125,2.034375,2.013750,0.256875


In [4]:
from pycaret.regression import *

In [5]:
setup(
    data=data,
    target="TorqueMeasured",
    session_id=123,
    ignore_features=["Time"]
)

,Description,Value
0,Session id,123
1,Target,TorqueMeasured
2,Target type,Regression
3,Original data shape,"(2500, 13)"
4,Transformed data shape,"(2500, 12)"
5,Transformed train set shape,"(1750, 12)"
6,Transformed test set shape,"(750, 12)"
7,Ignore features,1
8,Numeric features,11
9,Preprocess,True


In [6]:
best = compare_models()

predict_model(best)

save_model(best, "IPMSM_AutoML")

,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE,TT (Sec)
omp,Orthogonal Matching Pursuit,0.1179,0.0491,0.2201,0.0732,0.0827,0.0757,0.0220
ridge,Ridge Regression,0.1185,0.0491,0.2201,0.0726,0.0825,0.0757,0.0200
br,Bayesian Ridge,0.1184,0.0491,0.2203,0.0709,0.0826,0.0757,0.0230
lr,Linear Regression,0.1194,0.0492,0.2204,0.0693,0.0825,0.0761,1.2700
huber,Huber Regressor,0.1171,0.0495,0.2210,0.0658,0.0829,0.0752,0.0420
par,Passive Aggressive Regressor,0.1262,0.0504,0.2231,0.0472,0.0836,0.0802,0.0220
et,Extra Trees Regressor,0.1182,0.0507,0.2241,0.0277,0.0832,0.0740,0.1980
en,Elastic Net,0.1196,0.0531,0.2294,-0.0070,0.0849,0.0751,0.0210
lasso,Lasso Regression,0.1196,0.0531,0.2294,-0.0070,0.0849,0.0751,0.9990
llar,Lasso Least Angle Regression,0.1196,0.0531,0.2294,-0.0070,0.0849,0.0751,0.0210


,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE
0,Orthogonal Matching Pursuit,0.1228,0.0537,0.2318,-0.0082,0.0897,0.0839


Transformation Pipeline and Model Successfully Saved


(Pipeline(memory=Memory(location=None),
          steps=[('numerical_imputer',
                  TransformerWrapper(include=['TorqueCmd1', 'TorqueEst1', 'Iq2',
                                              'Speed', 'Id1', 'Iq1',
                                              'TorqueCmd2', 'TorqueEst2', 'Vdc1',
                                              'Vdc2', 'Id2'],
                                     transformer=SimpleImputer())),
                 ('categorical_imputer',
                  TransformerWrapper(include=[],
                                     transformer=SimpleImputer(strategy='most_frequent'))),
                 ('trained_model', OrthogonalMatchingPursuit())]),
 'IPMSM_AutoML.pkl')